In [1]:
using LowLevelFEM

[ Info: Precompiling LowLevelFEM [6171b9fb-adbf-4751-adb9-5faded75de07](cache misses: include_dependency fsize change (2), incompatible header (7))
[ Info: Precompiling LowLevelFEM [6171b9fb-adbf-4751-adb9-5faded75de07] (cache misses: include_dependency fsize change (4), incompatible header (14))

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


In [2]:
openGeometry("periodic-3D.geo")
#openPreProcessor()

In [3]:
mat = Material("body")

U = Field([mat], type=:VectorField, dim=3, fieldName=:u);
Φ = Field([mat], type=:VectorField, dim=3, fieldName=:φ);

In [4]:
Ku = ∫(ε(U)' ⋅ D(:Solid, mat) ⋅ ε(U));

In [5]:
bc_u = BoundaryCondition("P", ux=0, uy=0, uz=0, field=U)
bc_φ = BoundaryCondition("P", φx=0.01, φy=0, φz=0, field=Φ)


periodic1 = MPC(master="Q", slave="P", field=U)
periodic2 = MPC(master="Q", slave="P", field=Φ)
mpc_u1 = MPC(master="P", slave="left", field=U)
mpc_φ1 = MPC(master="P", slave="left", field=Φ);
mpc_u2 = MPC(master="Q", slave="right", field=U)
mpc_φ2 = MPC(master="Q", slave="right", field=Φ);

In [6]:
R = rigidRotationMap(mpc_u1, mpc_φ1);
R += rigidRotationMap(mpc_u2, mpc_φ2);

In [7]:
Kuφ = Ku * R
Kφ = R' * Ku * R

K = SystemMatrix([Ku Kuφ; Kuφ' Kφ]);

In [8]:
fu = ∫(U ⋅ [0,0,0])
fφ = ∫(Φ ⋅ [0,0,0], Γ="P")

F = SystemVector([fu, fφ]);

In [9]:
u, φ = solveField(K, F, support=[bc_u, bc_φ], mpc=[mpc_u1, mpc_φ1, mpc_u2, mpc_φ2, periodic1, periodic2])

(VectorField(Matrix{Float64}[], [0.0; 0.0; … ; -0.04001184161012045; 0.022146126545469293;;], [0.0], Int64[], 1, :v3D, Problem("periodic-3D", :VectorField, 3, 3, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 1738, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :rhs, false)), VectorField(Matrix{Float64}[], [0.01; 0.0; … ; 0.0; 0.0;;], [0.0], Int64[], 1, :v3D, Problem("periodic-3D", :VectorField, 3, 3, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 1738, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :φ, :rhs, false)))

In [10]:
showDoFResults(u + R * φ, name="u", visible=true, factor=20)

0

In [11]:
openPostProcessor()

XOpenIM() failed
Fontconfig warning: using without calling FcInit()
